# 01_bulk_width_gate: A100 Colab queue runner

P1 bulk baseline and W1 transverse-width gate. This notebook runs one immutable trajectory shard from the audited production matrix. Production mode uses the random serial schedule, 25 independent records assigned in five-trajectory shards, exactly `2*Ny` cycles, complex128, and no physical burn-in. Observables are streamed; covariances are transient scratch objects rather than permanent ensemble data.

Upload this entire numbered folder to `MyDrive`, select an A100 GPU, and run all cells. The resumable queue checks active Drive storage before every new shard and skips archives that already passed checksum verification. It defaults to the separate, one-shard-per-case calibration pilot; production must be selected explicitly.

## 1. Mount Drive and locate the self-contained bundle


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
from datetime import datetime, timezone
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = '01_bulk_width_gate'
MYDRIVE = Path('/content/drive/MyDrive')
candidates = [
    MYDRIVE / BUNDLE_NAME,
    MYDRIVE / 'final_production_ready_figure_scripts' / BUNDLE_NAME,
]
BUNDLE_ROOT = next((p for p in candidates if (p / 'run_bundle.py').exists()), None)
if BUNDLE_ROOT is None:
    matches = list(MYDRIVE.glob(f'*/{BUNDLE_NAME}/run_bundle.py'))
    if len(matches) == 1:
        BUNDLE_ROOT = matches[0].parent
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Upload the complete {BUNDLE_NAME} folder to MyDrive')
print(f'[bundle] {BUNDLE_ROOT}')


## 2. Configure the resumable pilot or production queue


In [ ]:
RUN_PROFILE = 'pilot_calibration'  # pilot_calibration, pilot_science, or production
if RUN_PROFILE not in ('pilot_calibration', 'pilot_science', 'production'):
    raise ValueError(f'unknown RUN_PROFILE {RUN_PROFILE!r}')
PILOT_PLAN_PATH = BUNDLE_ROOT / 'pilot_plan.json'
pilot_plan = json.loads(PILOT_PLAN_PATH.read_text())
MODE = 'production' if RUN_PROFILE == 'production' else 'pilot'
PILOT_LEVEL = RUN_PROFILE.removeprefix('pilot_') if MODE == 'pilot' else None
PILOT_WIDTH = int(pilot_plan['provisional_width'])
profile_case_ids = None if MODE == 'production' else list(pilot_plan[PILOT_LEVEL][BUNDLE_NAME])
profile_default_shards = None if MODE == 'production' else list(pilot_plan['shard_indices'])
profile_case_shards = {} if MODE == 'production' else dict(pilot_plan.get('profile_case_shard_indices', {}).get(PILOT_LEVEL, {}).get(BUNDLE_NAME, {}))
CASE_IDS = None  # None uses the profile list; an explicit list replaces it
SHARD_INDICES = None  # None uses case-aware profile shards; an explicit list replaces them
CASE_IDS = profile_case_ids if CASE_IDS is None else CASE_IDS
CASE_PREFIXES = ()         # e.g. ('S2_',); empty means no prefix filter
MAX_CASES = None           # optional restart/debug cap; None means the full queue
RUN_QUEUE = True
CREDITS_PER_A100_HOUR = float(pilot_plan['credit_rate_per_a100_hour'])
profile_hour_caps = {'pilot_calibration': 7, 'pilot_science': 32, 'production': None}
MAX_SESSION_GPU_HOURS_OVERRIDE = None
MAX_SESSION_GPU_HOURS = profile_hour_caps[RUN_PROFILE] if MAX_SESSION_GPU_HOURS_OVERRIDE is None else MAX_SESSION_GPU_HOURS_OVERRIDE
DRIVE_WORKING_LIMIT_GB = 12.0
DRIVE_ABSOLUTE_EDGE_GB = 14.0
REQUIRED_HEADROOM_GB = 1.0
ACCEPTED_WIDTH_JSON = '/content/drive/MyDrive/classA_final_production_outputs/01_bulk_width_gate/accepted_width.json'
M3_BULK_GATE_JSON = '/content/drive/MyDrive/classA_final_production_outputs/05_scans_and_controls/m3_bulk_gate.json'
GATE_DECISIONS_JSON = '/content/drive/MyDrive/classA_final_production_outputs/01_bulk_width_gate/core_gate_decisions.json'
DRIVE_ROOT = MYDRIVE

OUTPUT_COLLECTION = 'classA_pilot_outputs' if MODE == 'pilot' else 'classA_final_production_outputs'
OUTPUT_ROOT = MYDRIVE / OUTPUT_COLLECTION
config = json.loads((BUNDLE_ROOT / 'production_config.json').read_text())
print(json.dumps(config['locked_contract'], indent=2))
print({'run_profile': RUN_PROFILE, 'mode': MODE, 'cases': len(CASE_IDS or []), 'explicit_shard_override': SHARD_INDICES, 'default_profile_shards': profile_default_shards, 'case_specific_profile_shards': profile_case_shards, 'session_hour_cap': MAX_SESSION_GPU_HOURS, 'output': str(OUTPUT_ROOT)})


## 3. Resolve the currently eligible case queue


In [ ]:
base = [sys.executable, '-u', str(BUNDLE_ROOT / 'run_bundle.py'), '--drive-root', str(DRIVE_ROOT)]
pilot_args = ([] if False or MODE != 'pilot' else ['--pilot-width', str(PILOT_WIDTH)])
width_args = ['--accepted-width-json', ACCEPTED_WIDTH_JSON] if False else []
m3_args = (['--m3-bulk-gate-json', M3_BULK_GATE_JSON] if False and Path(M3_BULK_GATE_JSON).exists() else [])
launch_args = (['--gate-decisions-json', GATE_DECISIONS_JSON] if False else [])
listed = subprocess.run([*base, '--mode', MODE, *pilot_args, *width_args, *m3_args, *launch_args, '--list-cases-json'], check=True, text=True, capture_output=True)
case_rows = json.loads(listed.stdout)
case_ids = [row['case_id'] for row in case_rows]
shard_counts = {row['case_id']: int(row['shard_count']) for row in case_rows}
def selected_shards_for_case(case_id):
    if SHARD_INDICES is not None:
        return list(SHARD_INDICES)
    if MODE == 'production':
        return list(range(shard_counts[case_id]))
    return list(profile_case_shards.get(case_id, profile_default_shards))
gate_requirements = config.get('launch_gate_requirements', {})
blocked_by_gate = {}
if gate_requirements and MODE == 'production':
    gate_path = Path(GATE_DECISIONS_JSON)
    if not gate_path.exists():
        raise FileNotFoundError(f'create the evidence-backed launch decision file: {gate_path}')
    gate_payload = json.loads(gate_path.read_text())
    if gate_payload.get('schema_version') != 1 or gate_payload.get('status') != 'accepted':
        raise RuntimeError(f'launch decision file is not accepted: {gate_path}')
    decisions = gate_payload.get('decisions', {})
    for case in case_ids:
        for campaign, requirements in gate_requirements.items():
            prefix = campaign.replace('_', '') + '_'
            if case.startswith(prefix):
                failed = [name for name in requirements if decisions.get(name) is not True]
                if failed:
                    blocked_by_gate[case] = failed
                break
unknown = sorted(set(CASE_IDS or []) - set(case_ids))
if unknown:
    raise KeyError(f'unknown CASE_IDS: {unknown}')
queued_case_ids = list(case_ids if CASE_IDS is None else CASE_IDS)
explicitly_blocked = {case: blocked_by_gate[case] for case in queued_case_ids if case in blocked_by_gate}
if CASE_IDS is not None and explicitly_blocked:
    raise RuntimeError(f'explicit CASE_IDS are blocked by launch gates: {explicitly_blocked}')
queued_case_ids = [case for case in queued_case_ids if case not in blocked_by_gate]
if CASE_PREFIXES:
    queued_case_ids = [case for case in queued_case_ids if case.startswith(tuple(CASE_PREFIXES))]
if MAX_CASES is not None:
    queued_case_ids = queued_case_ids[:int(MAX_CASES)]
if not queued_case_ids:
    raise ValueError('the selected case queue is empty')
if SHARD_INDICES is not None and not SHARD_INDICES:
    raise ValueError('SHARD_INDICES is empty')
print(f'{len(case_ids)} listed cases; {len(blocked_by_gate)} gate-blocked; {len(queued_case_ids)} queued')
print('\n'.join(queued_case_ids))


## 4. Run the guarded, restart-safe case and shard queue


In [ ]:
guard_base = [
    sys.executable, str(BUNDLE_ROOT / 'src' / 'drive_storage_guard.py'),
    '--output-root', str(OUTPUT_ROOT),
    '--working-limit-gb', str(DRIVE_WORKING_LIMIT_GB),
    '--absolute-edge-gb', str(DRIVE_ABSOLUTE_EDGE_GB),
    '--required-headroom-gb', str(REQUIRED_HEADROOM_GB),
]
def require_storage_clear():
    print(f'[{datetime.now(timezone.utc).isoformat()}] checking Drive storage: {OUTPUT_ROOT}')
    subprocess.run(guard_base, check=True)

session_started = time.monotonic()
processed_shards = 0
shard_wall_seconds = []
selected_total_shards = sum(len(selected_shards_for_case(case)) for case in queued_case_ids)
def require_time_budget():
    elapsed_hours = (time.monotonic() - session_started) / 3600.0
    if MAX_SESSION_GPU_HOURS is not None and elapsed_hours >= float(MAX_SESSION_GPU_HOURS):
        raise RuntimeError(f'session GPU-hour cap reached ({elapsed_hours:.3f} h); restart only after reviewing saved timings')
    return elapsed_hours

def progress_report(label, item_seconds=None):
    global processed_shards
    if item_seconds is not None:
        shard_wall_seconds.append(float(item_seconds))
        processed_shards += 1
    elapsed_h = (time.monotonic() - session_started) / 3600.0
    credits = elapsed_h * CREDITS_PER_A100_HOUR
    mean_s = sum(shard_wall_seconds) / len(shard_wall_seconds) if shard_wall_seconds else None
    queue_remaining = max(0, selected_total_shards - processed_shards)
    queue_eta_h = None if mean_s is None else mean_s * queue_remaining / 3600.0
    remaining_h = None if MAX_SESSION_GPU_HOURS is None else max(0.0, float(MAX_SESSION_GPU_HOURS) - elapsed_h)
    print({'progress': label, 'processed_shards': processed_shards, 'total_shards': selected_total_shards, 'queue_shards_remaining': queue_remaining, 'rolling_queue_eta_hours': None if queue_eta_h is None else round(queue_eta_h, 3), 'session_hours': round(elapsed_h, 4), 'session_credits': round(credits, 2), 'mean_shard_minutes': None if mean_s is None else round(mean_s / 60.0, 2), 'session_cap_hours_remaining': None if remaining_h is None else round(remaining_h, 3), 'utc': datetime.now(timezone.utc).isoformat()})

if not RUN_QUEUE:
    raise RuntimeError('RUN_QUEUE is False; inspect the resolved queue, then set it to True')
progress_report('queue_start')
total_queue_shards = sum(len(selected_shards_for_case(case)) for case in queued_case_ids)
queue_bar = tqdm(total=total_queue_shards, desc=f'{BUNDLE_NAME} queue', unit='shard')
for selected_case in tqdm(queued_case_ids, desc=f'{BUNDLE_NAME} cases', unit='case'):
    case_shards = selected_shards_for_case(selected_case)
    invalid = [index for index in case_shards if not 0 <= int(index) < shard_counts[selected_case]]
    if invalid:
        raise IndexError(f'{selected_case} has {shard_counts[selected_case]} shards; invalid selection: {invalid}')
    for shard_index in tqdm(case_shards, desc=selected_case, unit='shard', leave=False):
        require_time_budget()
        require_storage_clear()
        preflight_cmd = [*base, '--mode', MODE, *pilot_args, '--case-id', selected_case, '--shard-index', str(shard_index), *width_args, *m3_args, *launch_args, '--preflight-only']
        print(f'\n[{datetime.now(timezone.utc).isoformat()}] selected case={selected_case}, shard={shard_index}; queue={processed_shards + 1}/{total_queue_shards}')
        subprocess.run(preflight_cmd, check=True)
        run_cmd = [*base, '--mode', MODE, *pilot_args, '--case-id', selected_case, '--shard-index', str(shard_index), *width_args, *m3_args, *launch_args]
        print(f'[launch] {" ".join(run_cmd)}')
        item_started = time.monotonic()
        subprocess.run(run_cmd, check=True)
        item_elapsed = time.monotonic() - item_started
        queue_bar.update(1)
        progress_report(f'completed_or_verified {selected_case} shard {shard_index}', item_elapsed)
queue_bar.close()

elapsed_hours = (time.monotonic() - session_started) / 3600.0
progress_report('queue_finished')
print(f'[session] {elapsed_hours:.3f} A100 h, {elapsed_hours * CREDITS_PER_A100_HOUR:.2f} credits')


## 5. Analyze a completed gate matrix

Run this only after every required case and all five shards are archived.

In [ ]:
RUN_GATE_ANALYSIS = False
GATE_NAME = 'width'
GATE_OUTPUT = MYDRIVE / 'classA_final_production_outputs' / BUNDLE_NAME / 'accepted_width.json'
if RUN_GATE_ANALYSIS:
    if MODE != 'production':
        raise RuntimeError('gate analysis is forbidden for non-gating pilot archives')
    archive_root = MYDRIVE / 'classA_final_production_outputs' / BUNDLE_NAME
    subprocess.run([sys.executable, str(BUNDLE_ROOT / 'src' / 'gate_analysis.py'), GATE_NAME, '--archive-root', str(archive_root), '--output', str(GATE_OUTPUT)], check=True)
    print(GATE_OUTPUT.read_text())


## Final. Disconnect the Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
